# GPU Monte Carlo — Colab Dev Notebook

**Project:** GPU-Accelerated Monte Carlo Option Pricing  
**Course:** BLG 562E Spring 2026  
**Team:** Talha Sarlık, Mehmet Bedirhan Önder

Use this notebook on Google Colab. Set runtime to GPU (T4 free, A100 Pro).

## 1. GPU sanity check

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv
!nvcc --version
!ncu --version | head -1
!nsys --version | head -1

## 2. Clone repo (or pull latest)

In [ ]:
import os
REPO_URL = 'https://github.com/<your-org>/gpu-monte-carlo.git'  # TODO: set this
DEST = '/content/project'
if not os.path.isdir(DEST):
    !git clone $REPO_URL $DEST
else:
    %cd $DEST
    !git pull
%cd $DEST

## 3. Build everything

On A100 use `make ARCH=sm_80 all` instead.

In [ ]:
!make clean && make all

## 4. Quick sanity run

In [ ]:
!./build/mc_cpu --paths 100000
!./build/mc_naive --paths 1000000
!./build/mc_shared --paths 1000000
!./build/mc_antithetic --paths 1000000

## 5. Validate correctness (must pass before benchmarking)

In [ ]:
!python tests/validate.py

## 6. Full benchmark sweep

In [ ]:
!bash benchmarks/run_all.sh

## 7. Plot results

In [ ]:
!python benchmarks/plot.py
from IPython.display import Image, display
for p in ['results/throughput.png', 'results/speedup.png', 'results/convergence.png']:
    display(Image(p))

## 8. Profile a kernel with Nsight Compute

In [ ]:
!ncu --set full -o results/mc_naive_full ./build/mc_naive --paths 10000000
!ncu --print-summary per-kernel ./build/mc_shared --paths 10000000
!ncu --print-summary per-kernel ./build/mc_antithetic --paths 10000000

## 9. Save results to Google Drive (persist across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cuda-project/results
!cp -r results/* /content/drive/MyDrive/cuda-project/results/